In [3]:
from sympy.codegen.ast import integer

with open("the-verditct.txt", "r", encoding="utf-8") as file:
    raw_text=file.read()
print("Total nr of chars: ",len(raw_text))
print(raw_text[:99])

Total nr of chars:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [6]:
import re
text="Hello, World. This is a test."
result=re.split(r"([.,]|\s)",text)
result=[c for c in result if c.strip()]
print(result)

['Hello', ',', 'World', '.', 'This', 'is', 'a', 'test', '.']


In [8]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [11]:
preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [19]:
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
print(vocab_size)

vocab={token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=50:
        break;

1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [21]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed=[ item.strip() for item in preprocessed if item.strip() ]
        ids=[self.str_to_int[w] for w in preprocessed]
        return ids

    def decode(self,ids):
        text=" ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [24]:
tokenizer=SimpleTokenizerV1(vocab)
text=""""It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)

back_to_text=tokenizer.decode(ids)
print(back_to_text)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [25]:
text2="Hello, do you like tea?"
print(tokenizer.encode(text2))

KeyError: 'Hello'

In [35]:
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])

vocab={token:integer for integer,token in enumerate(all_tokens)}
print(len(vocab))

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [36]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int=vocab
        self.int_to_str={i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed=[ item.strip() for item in preprocessed if item.strip() ]
        preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids=[self.str_to_int[w] for w in preprocessed]
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [96]:
text1="Hello, do you like tea?"
text2="In the sunlit terraces of the palace."
text=" <|endoftext|> ".join((text1,text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [43]:
tokenizer=SimpleTokenizerV2(vocab)
ids=tokenizer.encode(text)
print(ids)

print(tokenizer.decode(ids))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [45]:
import tiktoken
from importlib.metadata import version

tiktoken.__version__

'0.13.0'

In [70]:
tokenizer=tiktoken.get_encoding("gpt2")

text = (
"Hello, do you like tea? <|endoftext|> In the sunlit terraces"
"of someunknownPlace."
)

numbers=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(numbers)

print(tokenizer.decode(numbers))

text2="pula mea mare"
print(tokenizer.encode(text2))
print(tokenizer.decode([533]))

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.
[79, 4712, 502, 64, 285, 533]
are


In [75]:
with open("the-verditct.txt","r") as file:
    raw_text=file.read()

encoded_text=tokenizer.encode(raw_text)
print(len(encoded_text))

enc_sample=encoded_text[50:]

5145


In [78]:
context_size=4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"{x}")
print(f"     {y}")

[290, 4920, 2241, 287]
     [4920, 2241, 287, 257]


In [83]:
for i in range(1,10):
    context=enc_sample[:i]
    desired=enc_sample[i]
    print(f"{context}: ----> {desired}")
    print(f"{tokenizer.decode(context)}: ----> {tokenizer.decode([desired])}")

[290]: ----> 4920
 and: ---->  established
[290, 4920]: ----> 2241
 and established: ---->  himself
[290, 4920, 2241]: ----> 287
 and established himself: ---->  in
[290, 4920, 2241, 287]: ----> 257
 and established himself in: ---->  a
[290, 4920, 2241, 287, 257]: ----> 4489
 and established himself in a: ---->  vill
[290, 4920, 2241, 287, 257, 4489]: ----> 64
 and established himself in a vill: ----> a
[290, 4920, 2241, 287, 257, 4489, 64]: ----> 319
 and established himself in a villa: ---->  on
[290, 4920, 2241, 287, 257, 4489, 64, 319]: ----> 262
 and established himself in a villa on: ---->  the
[290, 4920, 2241, 287, 257, 4489, 64, 319, 262]: ----> 34686
 and established himself in a villa on the: ---->  Riv


In [89]:
import torch
from torch.utils.data import DataLoader,Dataset

class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids=tokenizer.encode(txt)
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1: i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, item):
        return self.input_ids[item], self.target_ids[item]

In [97]:
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True, num_worker=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)

    dataloader=DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_worker,
        drop_last=drop_last
    )
    return dataloader

In [98]:
with open("the-verditct.txt","r") as file:
    raw_text=file.read()

dataloader=create_dataloader_v1(raw_text,3,6,3,shuffle=False)
data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

[tensor([[   40,   367,  2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402,   271, 10899],
        [  402,   271, 10899,  2138,   257,  7026]]), tensor([[  367,  2885,  1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271, 10899,  2138],
        [  271, 10899,  2138,   257,  7026, 15632]])]


In [100]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [106]:
input_ids=torch.tensor([2,3,5,1])
vocab_size=6
output_dim=3

torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(vocab_size,output_dim)
print(embedding_layer.weight)

print(embedding_layer(torch.tensor([0])))

print(embedding_layer(input_ids))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[ 0.3374, -0.1778, -0.1690]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [117]:
vocab_size=50257
output_dim=256

token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

max_length=4
dataloader=create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=4,shuffle=False)
data_iter=iter(dataloader)
inputs,targets=next(data_iter)
print(inputs)
print(targets)
print(tokenizer.decode(inputs[0].tolist()))
print(tokenizer.decode(targets[0].tolist()))

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
I HAD always
 HAD always thought


In [120]:
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)
print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 0.9711, -0.0824, -1.3424,  ..., -1.1325,  0.5965,  0.3273],
         [ 0.0626, -0.3465, -1.3466,  ..., -0.1176,  0.7653, -1.2771],
         [-0.0305,  1.3181, -0.3010,  ..., -0.8743,  0.3046, -1.2370],
         [ 1.1920,  0.2388, -0.5406,  ..., -0.1861, -0.2685,  0.9379]],

        [[ 0.4604, -0.7661, -0.5669,  ...,  1.4547, -0.9964,  0.4565],
         [-1.5673, -0.7319, -2.2110,  ..., -0.3470,  0.1747, -0.5825],
         [ 0.3443,  0.6836,  0.5774,  ...,  0.1363,  0.3300,  0.5801],
         [ 0.3800, -1.3351,  0.1048,  ..., -0.1080, -0.4211, -0.6672]],

        [[ 1.5713,  1.0521,  0.1572,  ..., -0.6151, -0.0403,  1.3856],
         [ 0.8081, -0.0830,  0.0368,  ..., -0.1944, -0.1845,  0.9693],
         [ 0.2332, -0.2321, -0.5273,  ...,  0.4290, -0.1104, -1.1548],
         [ 1.8497, -0.7649, -2.9431,  ...,  1.8072,  0.2904,  0.8767]],

        ...,

        [[ 0.9453, -0.5761,  1.9967,  ..., -0.6514, -1.1277,  0.0557],
         [-1.0662,  1.0915,  0.61

In [121]:
context_length=max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)
pos_embeddings=pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)
print(pos_embeddings)

torch.Size([4, 256])
tensor([[ 0.8737, -0.1034, -0.2278,  ...,  0.8780, -1.2694,  0.3087],
        [ 0.4584, -1.2803, -2.3941,  ..., -0.9010,  0.3083, -0.5395],
        [ 0.8984,  1.5400,  0.5817,  ...,  0.7219,  0.9066, -0.4481],
        [ 0.5465,  1.0353,  0.3921,  ...,  0.3875,  0.0720,  0.4475]],
       grad_fn=<EmbeddingBackward0>)


In [122]:
input_embeddings=token_embeddings+pos_embeddings
print(input_embeddings.shape)
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 1.8448, -0.1858, -1.5701,  ..., -0.2545, -0.6729,  0.6361],
         [ 0.5211, -1.6268, -3.7406,  ..., -1.0186,  1.0737, -1.8166],
         [ 0.8679,  2.8581,  0.2808,  ..., -0.1524,  1.2111, -1.6852],
         [ 1.7385,  1.2741, -0.1485,  ...,  0.2014, -0.1965,  1.3854]],

        [[ 1.3341, -0.8695, -0.7947,  ...,  2.3327, -2.2657,  0.7653],
         [-1.1088, -2.0122, -4.6051,  ..., -1.2480,  0.4831, -1.1221],
         [ 1.2427,  2.2236,  1.1592,  ...,  0.8583,  1.2365,  0.1319],
         [ 0.9265, -0.2998,  0.4969,  ...,  0.2794, -0.3492, -0.2196]],

        [[ 2.4450,  0.9486, -0.0706,  ...,  0.2629, -1.3097,  1.6944],
         [ 1.2665, -1.3633, -2.3572,  ..., -1.0955,  0.1238,  0.4298],
         [ 1.1316,  1.3079,  0.0544,  ...,  1.1509,  0.7961, -1.6029],
         [ 2.3963,  0.2704, -2.5510,  ...,  2.1947,  0.3624,  1.3242]],

        ...,

        [[ 1.8190, -0.6796,  1.7689,  ...,  0.2266, -2.3971,  0.3645],
         [-0.6078, -0.1888, -1.78